# TF-IDF baseline для общей классификации резюме

Ноутбук используется для обучения baseline-модели классификации резюме на основе `TF-IDF` и `LogisticRegression`.

---

В этом ноутбуке используется датасет `resume_dataset_general_grouped.csv`.

Он создается из базового обработанного датасета `resume_dataset.csv` с помощью скрипта:
>scripts/prepare_general_grouped_dataset.py

## Импорты и настройка путей

In [15]:
from pathlib import Path

import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split


BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data" / "processed"
DATA_PATH = DATA_DIR / "resume_dataset_general_grouped.csv"

MODELS_DIR = BASE_DIR / "app" / "models"
VECTORIZER_PATH = MODELS_DIR / "tfidf_vectorizer.pkl"
CLASSIFIER_PATH = MODELS_DIR / "category_classifier.pkl"

TEXT_COLUMN = "resume_text"
TARGET_COLUMN = "target_role"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Dataset path:", DATA_PATH)
print("Models directory:", MODELS_DIR)


Base directory: /content
Dataset path: /content/data/processed/resume_dataset_general_grouped.csv
Models directory: /content/app/models


## Загрузка датасета

In [16]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (2481, 4)


,resume_text,target_role,source,original_category
0,INFORMATION TECHNOLOGY SPECIALIST(DISCOUNTPCFI...,TECHNICAL,huggingface_darshan_04_resume_classification_g...,INFORMATION-TECHNOLOGY
1,BUSINESS DEVELOPMENT DIRECTOR Summary ...,BUSINESS,huggingface_darshan_04_resume_classification_g...,BUSINESS-DEVELOPMENT
2,ENGINEERING INTERN Personal Summary ...,TECHNICAL,huggingface_darshan_04_resume_classification_g...,ENGINEERING
3,ENGINEERING LAB TECHNICIAN Summary To...,TECHNICAL,huggingface_darshan_04_resume_classification_g...,ENGINEERING
4,GUEST LECTURER Accomplishments ...,PEOPLE,huggingface_darshan_04_resume_classification_g...,FITNESS


## Базовая очистка данных

- оставляются только текст резюме и целевая категория;
- удаляются строки с пропусками;
- значения приводятся к строковому типу;
- удаляются слишком короткие тексты;
- удаляются дубликаты по паре `resume_text` и `target_role`.


In [17]:
df = df[[TEXT_COLUMN, TARGET_COLUMN]].dropna()

df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(str).str.strip()

df = df[df[TEXT_COLUMN].str.len() > 100]
df = df.drop_duplicates(subset=[TEXT_COLUMN, TARGET_COLUMN])

print("Dataset shape after cleaning:", df.shape)
print("\nClass distribution:")
df[TARGET_COLUMN].value_counts()


Dataset shape after cleaning: (2481, 2)

Class distribution:


,count
target_role,
BUSINESS,461
PEOPLE,444
CREATIVE,403
FINANCE,350
OPERATIONS,327
TECHNICAL,238
SERVICE,140
LEGAL,118


## Разделение на train и test

In [18]:
X = df[TEXT_COLUMN]
y = df[TARGET_COLUMN]

min_class_count = y.value_counts().min()
stratify = y if min_class_count >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=stratify,
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])


Train size: 1984
Test size: 497


## TF-IDF vectorizer и классификатор


In [19]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)

classifier = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

print(vectorizer)
print(classifier)

TfidfVectorizer(max_df=0.95, max_features=50000, min_df=2, ngram_range=(1, 2),
                stop_words='english')
LogisticRegression(class_weight='balanced', max_iter=3000, n_jobs=-1,
                   random_state=42)


## Обучение baseline-модели

In [20]:
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Train TF-IDF shape:", X_train_vec.shape)
print("Test TF-IDF shape:", X_test_vec.shape)

classifier.fit(X_train_vec, y_train)

Train TF-IDF shape: (1984, 50000)
Test TF-IDF shape: (497, 50000)


LogisticRegression(class_weight='balanced', max_iter=3000, n_jobs=-1,
                   random_state=42)

## Оценка качества

Используются метрики:

- `accuracy`
- `macro_f1`
- `weighted_f1`

In [21]:
y_pred = classifier.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")
precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)

metrics = {
    "model_name": "TF-IDF + LogisticRegression baseline",
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "precision_macro": precision_macro,
    "recall_macro": recall_macro,
}

print("Metrics:")
print(f"accuracy:        {accuracy:.4f}")
print(f"macro_f1:        {macro_f1:.4f}")
print(f"weighted_f1:     {weighted_f1:.4f}")
print(f"precision_macro: {precision_macro:.4f}")
print(f"recall_macro:    {recall_macro:.4f}")

metrics

Metrics:
accuracy:        0.6982
macro_f1:        0.6867
weighted_f1:     0.6958
precision_macro: 0.6972
recall_macro:    0.7028


{'model_name': 'TF-IDF + LogisticRegression baseline',
 'accuracy': 0.6981891348088531,
 'macro_f1': 0.686692962685991,
 'weighted_f1': 0.6957991658647903,
 'precision_macro': 0.6972304288064117,
 'recall_macro': 0.7027984156841427}

## Classification report


In [22]:
print(classification_report(y_test, y_pred, zero_division=0))

              precision    recall  f1-score   support

    BUSINESS       0.69      0.67      0.68        92
    CREATIVE       0.79      0.46      0.58        81
     FINANCE       0.83      0.79      0.81        70
       LEGAL       0.42      0.58      0.49        24
  OPERATIONS       0.75      0.83      0.79        65
      PEOPLE       0.70      0.70      0.70        89
     SERVICE       0.83      0.68      0.75        28
   TECHNICAL       0.57      0.92      0.70        48

    accuracy                           0.70       497
   macro avg       0.70      0.70      0.69       497
weighted avg       0.72      0.70      0.70       497



## Сохранение baseline-модели

После обучения сохраняются два артефакта:

- `tfidf_vectorizer.pkl`
- `category_classifier.pkl`


In [23]:
joblib.dump(vectorizer, VECTORIZER_PATH)
joblib.dump(classifier, CLASSIFIER_PATH)

print(f"Vectorizer saved to: {VECTORIZER_PATH}")
print(f"Classifier saved to: {CLASSIFIER_PATH}")

Vectorizer saved to: /content/app/models/tfidf_vectorizer.pkl
Classifier saved to: /content/app/models/category_classifier.pkl
